# 05 — Capacity & Accessibility Analysis

This notebook takes the shortlisted opportunity areas and performs two final feasibility checks:

1. **Capacity:** compare local average daily demand with an assumed dark-store capacity.
2. **Geographic accessibility:** use customer-to-opportunity distance as an initial geographic accessibility proxy.

Important: the **1,200 orders/day/store capacity** is an analytical planning assumption for this case study. The **10-minute SLA target is illustrative**, and geographic distance is not treated as equivalent to delivery time.

In [ ]:
import pandas as pd
import numpy as np

ORDERS_PATH = "../outputs/customer_orders_with_clusters.csv"
OPPORTUNITY_PATH = "../outputs/opportunity_areas.csv"

orders = pd.read_csv(ORDERS_PATH)
opportunity = pd.read_csv(OPPORTUNITY_PATH)

orders["Order_Timestamp"] = pd.to_datetime(orders["Order_Timestamp"], errors="coerce")

print("Orders:", orders.shape)
print("Opportunity areas:", opportunity.shape)

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

analysis_start = orders["Order_Timestamp"].min().normalize()
analysis_end = orders["Order_Timestamp"].max().normalize()
analysis_days = (analysis_end - analysis_start).days + 1

print("Analysis start:", analysis_start.date())
print("Analysis end:", analysis_end.date())
print("Analysis days:", analysis_days)

## Capacity Check

The project uses:

**Average daily local demand = orders within 2 km / number of analysis days**

**Stores required = ceiling(average daily local demand / 1,200)**

**Capacity utilization = average daily local demand / 1,200 × 100**

This is a screening calculation, not a universal dark-store capacity benchmark.

In [ ]:
STORE_CAPACITY = 1200
CATCHMENT_KM = 2.0

results = []

for _, row in opportunity.sort_values("priority_rank").head(5).iterrows():
    d = haversine_km(
        row["area_lat"], row["area_lon"],
        orders["Customer_Lat"].to_numpy(),
        orders["Customer_Lon"].to_numpy()
    )

    local_orders = int((d <= CATCHMENT_KM).sum())
    avg_daily_local_orders = local_orders / analysis_days

    stores_required = int(np.ceil(avg_daily_local_orders / STORE_CAPACITY))
    capacity_utilization_pct = (avg_daily_local_orders / STORE_CAPACITY) * 100

    results.append({
        "opportunity_id": row["opportunity_id"],
        "cluster": row["cluster"],
        "area_lat": row["area_lat"],
        "area_lon": row["area_lon"],
        "opportunity_score": row["opportunity_score"],
        "competitors_within_2km": row["competitors_within_2km"],
        "nearest_competitor_km": row["nearest_competitor_km"],
        "local_orders_2km": local_orders,
        "avg_daily_local_orders": avg_daily_local_orders,
        "stores_required": stores_required,
        "capacity_utilization_pct": capacity_utilization_pct
    })

capacity_df = pd.DataFrame(results)
capacity_df

## Geographic Accessibility Check

For each shortlisted opportunity area, customer-to-area distances are summarized as:

- Average customer distance
- Percentage of orders within 1 km
- Percentage of orders within 2 km
- Maximum customer distance

These values are **geographic accessibility measures only**. They are not delivery-time measurements.

In [ ]:
accessibility_rows = []

for _, row in capacity_df.iterrows():
    d = haversine_km(
        row["area_lat"], row["area_lon"],
        orders["Customer_Lat"].to_numpy(),
        orders["Customer_Lon"].to_numpy()
    )

    local_d = d[d <= CATCHMENT_KM]

    if len(local_d) == 0:
        avg_distance = np.nan
        within_1km_pct = 0.0
        within_2km_pct = 0.0
        max_distance = np.nan
    else:
        avg_distance = float(local_d.mean())
        within_1km_pct = float((local_d <= 1.0).mean() * 100)
        within_2km_pct = float((local_d <= 2.0).mean() * 100)
        max_distance = float(local_d.max())

    accessibility_status = (
        "Good Geographic Accessibility"
        if within_1km_pct >= 50
        else "Needs Review"
    )

    accessibility_rows.append({
        **row.to_dict(),
        "avg_customer_distance_km": avg_distance,
        "orders_within_1km_pct": within_1km_pct,
        "orders_within_2km_pct": within_2km_pct,
        "max_customer_distance_km": max_distance,
        "accessibility_status": accessibility_status
    })

final_df = pd.DataFrame(accessibility_rows)
final_df.insert(0, "rank", np.arange(1, len(final_df) + 1))

final_df

## Final Shortlist

This table combines the opportunity score, competitor coverage, demand, capacity, and geographic accessibility into one interview-friendly view.

In [ ]:
final_df[[
    "rank", "opportunity_id", "cluster", "opportunity_score",
    "competitors_within_2km", "nearest_competitor_km",
    "avg_daily_local_orders", "stores_required",
    "capacity_utilization_pct", "avg_customer_distance_km",
    "orders_within_1km_pct", "orders_within_2km_pct",
    "accessibility_status"
]].sort_values("rank")

In [ ]:
final_df.to_csv("../outputs/final_opportunity_sla_analysis.csv", index=False)

print("Saved:", "../outputs/final_opportunity_sla_analysis.csv")

## Business Interpretation

The final output helps answer:

**1. Is demand high enough to justify considering the area?**  
Use the opportunity score and local average daily demand.

**2. Is competitor coverage relatively low?**  
Use competitor count within 2 km and nearest competitor distance.

**3. Can one assumed store handle the observed local demand?**  
Use stores required and capacity utilization.

**4. Is the area geographically close to its local customers?**  
Use average customer distance and the percentage of orders within 1 km.

### Important limitation

The case study does not have road-network travel-time, traffic, rider availability, actual company SLA, or store processing-time data. Therefore, it should not claim that a geographic radius directly guarantees a 10-minute delivery.